# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


1. What one row means for my lane: one row = one page, on one day, for one client (a page-day grain) — matching content_hash_id + client_hash_id + report_date in fact_content_daily_performance.

2. Table(s) I'll use: fact_content_daily_performance (the daily metrics), joined with dim_content for content-level attributes like content type.

3. Time window: month=2026-03 — a mid-panel month, not the sealed final month (2026-06), per the warning on this card.

4. What I'd predict or rank (label/proxy): same as my ML-03 framing — the gap between a page-day's actual CTR and its expected CTR given its average search position, used to rank pages by how much they're underperforming their position.

5. What I deliberately exclude: fact_content_query_90d (query-level data) — I'm working at the page-day grain for this contract, not the query grain, to keep the unit of analysis consistent with my ML-02/ML-03 work.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# Fact 1: grain check — confirm one row = one page-day
con.sql(f"""
SELECT COUNT(*) as total_rows,
       COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) as unique_combos
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────┐
│ total_rows │ unique_combos │
│   int64    │     int64     │
├────────────┼───────────────┤
│    9841378 │       9841378 │
└────────────┴───────────────┘

In [5]:
con.sql(f"""
DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) as rows_with_gsc_data,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND gsc_impressions > 0) as rows_with_real_impressions
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────────────┐
│ total_rows │ rows_with_gsc_data │ rows_with_real_impressions │
│   int64    │       int64        │           int64            │
├────────────┼────────────────────┼────────────────────────────┤
│    9841378 │            3611061 │                    3611061 │
└────────────┴────────────────────┴────────────────────────────┘

In [7]:
import pandas as pd

# Pull a working sample with our five features
trap_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_clicks,
    gsc_impressions,
    CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END as ctr,
    CASE WHEN gsc_impressions > 0 THEN gsc_sum_position * 1.0 / gsc_impressions ELSE NULL END as avg_position
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
""").df()

# Define label: is this row "underperforming" (below-median CTR for its position tier)?
trap_df['expected_ctr'] = trap_df['avg_position'].apply(lambda p: 0.787 if p <= 10 else 0.263)
trap_df['underperforming'] = (trap_df['ctr'] < trap_df['expected_ctr']).astype(int)

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# HONEST version: only real, pre-decision features
X_honest = trap_df[['gsc_impressions', 'avg_position']]
y = trap_df['underperforming']
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)
tree_honest = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_honest.fit(X_train, y_train)
print(f"Honest score: {tree_honest.score(X_test, y_test):.3f}")

# LEAKY version: add a column derived directly from the label
trap_df['ctr_minus_expected'] = trap_df['ctr'] - trap_df['expected_ctr']  # this literally encodes the label
X_leaky = trap_df[['gsc_impressions', 'avg_position', 'ctr_minus_expected']]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
tree_leaky = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_leaky.fit(X_train_l, y_train_l)
print(f"Leaky score (with label-derived column): {tree_leaky.score(X_test_l, y_test_l):.3f}")

# Remove the leaky column, keep the honest one
print("Leaky column removed. Keeping honest score as the real result.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest score: 0.999
Leaky score (with label-derived column): 1.000
Leaky column removed. Keeping honest score as the real result.


The trap: I added ctr_minus_expected, a column computed directly from the same values used to define the label. This inflated the score to 1.000 — pure leakage, as expected — and I removed it.

An honest observation: even without that column, my honest score was already 0.999. This is because my label (underperforming) is itself a hard threshold on avg_position, and avg_position was one of my two honest features — so the model could nearly perfectly learn the threshold rule directly. This is a milder form of the same leakage lesson: a label built too directly from one of your own features will always look artificially easy to predict, even without an explicit leaky column. A more honest label would need to avoid this direct dependency, or the "honest" score should be read with that caveat in mind.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT COUNT(*) as row_count,
       MIN(report_date) as earliest_date,
       MAX(report_date) as latest_date
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")


┌───────────┬───────────────┬─────────────┐
│ row_count │ earliest_date │ latest_date │
│   int64   │     date      │    date     │
├───────────┼───────────────┼─────────────┤
│   9841378 │ 2026-03-01    │ 2026-03-31  │
└───────────┴───────────────┴─────────────┘

What this data can never tell you:
1. Unbalanced history: per dim_clients, clients have very different gsc_data_start and ga4_data_start dates — some clients have over a year of history, others only a few months. Any pattern I find may be dominated by clients with longer history, not a universal truth.
2. GSC-only early rows: some clients only have has_gsc_access=true but has_ga4_access=false, meaning search performance data exists without engagement/analytics data for the same period.
3. Window overlaps: fact_content_query_90d uses a rolling 90-day window with sub-windows that overlap with the daily fact table's dates — combining both without care could create leakage between "past" and "future" periods.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.